In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
source_table = "retail_project.bronze.sales_raw"
target_table = "retail_project.silver.sales_cleaned"
error_table = "retail_project.silver.sales_errors"
checkpoint_path = "/Volumes/retail_project/bronze/sales_data_landing/_checkpoints/silver/"

In [0]:
def transform_sales_data(df):
    hash_cols = ["order_id", "cust_id", "event_ts"]
    return (
        df.drop("_rescued_data")
        .withColumn("product_name", F.when(F.col("product_name") == "Headphonz", "Headphones")
        .otherwise(F.col("product_name")))
        .withColumn("base_price", F.col("base_price").cast("double"))
        .withColumn("discount_amount", F.col("discount_amount").cast("double"))
        .withColumn("final_price", F.col("final_price").cast("double"))
        .withColumn("event_ts", F.to_timestamp(F.col("event_ts")))
        .withColumn("city", F.initcap(F.col("city")))
        .withColumn("cust_id", F.sha2(F.col("cust_id"), 256))
        .withColumn("row_hash", F.sha2(F.concat_ws("||", *hash_cols), 256))
        .withColumn("silver_processed_at", F.current_timestamp())
    )

In [0]:
def validate_data(df):
    is_valid_condition = (
        (F.col("final_price") >= 0) &
        (F.col("order_id").isNotNull()) &
        (F.col("event_ts").isNotNull())
    )
    valid_df = df.filter(is_valid_condition)
    invalid_df = df.filter(~is_valid_condition)
    
    return valid_df, invalid_df


In [0]:
def upsert_to_silver(batch_df, batch_id):
    cleaned_df = transform_sales_data(batch_df)

    valid_df, invalid_df = validate_data(cleaned_df)

    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (
            dt.alias("t")
            .merge(valid_df.alias("s"), "t.row_hash = s.row_hash")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        cleaned_df.write.format("delta").mode("append").saveAsTable(target_table)
    
    if invalid_df.count() > 0:
        invalid_df.write.format("delta").mode("append").saveAsTable(error_table)
        

In [0]:
save_df = (
    spark.readStream
    .table(source_table)
    .writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)
save_df.awaitTermination()